# TOOLBOXLAP — Google Colab Hugging Face / Ollama / ngrok API

One-cell educational/test launcher. Enable a GPU runtime, run the cell, enter a model (or press Enter for the default) and your ngrok authentication token, then copy the printed Base URL into Cline.

**Important:** Google Colab managed runtimes currently restrict certain public web-service/proxy uses. This notebook is intended as an educational test/demo; follow Colab's current terms and stop if the runtime blocks or terminates the workflow.

Website: https://toolboxlap.com  |  YouTube: https://www.youtube.com/@TOOLBOXLAP-u1c  |  GitHub: https://github.com/toolboxlap-ve/TOOLBOXLAP-Colab-Ollama


In [ ]:
# TOOLBOXLAP — Google Colab Hugging Face / Ollama / ngrok API
# One-cell educational/test launcher.
# IMPORTANT: Google Colab managed runtimes currently restrict certain public web-service/proxy uses.
# Use this for an educational test/demo and follow Colab's current terms.

import os
import sys
import time
import re
import shutil
import subprocess
import threading
import getpass
from urllib.parse import urlparse, unquote

DEFAULT_MODEL = "hf.co/HauhauCS/Qwen3.5-9B-Uncensored-HauhauCS-Aggressive:Q4_K_M"
PUBLIC_MODEL_ID = "toolboxlap"
OLLAMA_URL = "http://127.0.0.1:11434"
PROXY_PORT = 8000
HIGH_CONTEXT = 131072
FALLBACK_CONTEXT = 65536

print("=" * 72)
print("TOOLBOXLAP — Google Colab Hugging Face / Ollama / ngrok API")
print("=" * 72)

model_input = input(f"\nModel [ENTER = default: {DEFAULT_MODEL}]: ").strip()
MODEL = model_input or DEFAULT_MODEL

NGROK_AUTHTOKEN = getpass.getpass(
    "\nngrok authtoken (used only for this Colab session): "
).strip()
if not NGROK_AUTHTOKEN:
    raise ValueError("An ngrok authtoken is required.")

def sh(cmd, check=True, env=None, capture=False):
    print("+", " ".join(cmd), flush=True)
    return subprocess.run(
        cmd,
        check=check,
        text=True,
        env=env,
        capture_output=capture,
    )

def ensure_pkg(import_name, pip_name=None):
    try:
        __import__(import_name)
    except ImportError:
        sh([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])

def normalize_model(raw):
    value = raw.strip()
    if not value:
        return DEFAULT_MODEL
    if value.startswith("hf.co/"):
        return value

    parsed = urlparse(value)
    if parsed.netloc.lower() in {"huggingface.co", "www.huggingface.co"}:
        parts = [unquote(x) for x in parsed.path.split("/") if x]
        if len(parts) < 2:
            raise ValueError("Hugging Face URL must include owner and repository.")
        owner, repo = parts[0], parts[1]
        tag = ""
        if len(parts) >= 5 and parts[2] in {"blob", "resolve"}:
            filename = parts[-1]
            if filename.lower().endswith(".gguf"):
                tag = ":" + filename[:-5]
        q = re.search(r"(?:revision|tag)=([^&]+)", parsed.query)
        if q:
            tag = ":" + unquote(q.group(1))
        return f"hf.co/{owner}/{repo}{tag}"

    return value

MODEL = normalize_model(MODEL)
print(f"\nSelected backend model: {MODEL}", flush=True)

ensure_pkg("requests")
ensure_pkg("flask")

if shutil.which("zstd") is None:
    if shutil.which("apt-get") is None:
        raise RuntimeError("apt-get is unavailable; cannot install zstd.")
    sh(["apt-get", "update"])
    sh(["apt-get", "install", "-y", "zstd"])

if shutil.which("ollama") is None:
    sh(["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"])

if shutil.which("ollama") is None:
    raise RuntimeError("Ollama installation failed.")

import requests

def wait_http(url, timeout=90):
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        try:
            if requests.get(url, timeout=3).ok:
                return
        except Exception:
            pass
        time.sleep(1)
    raise TimeoutError(f"Timed out waiting for {url}")

def start_ollama(context):
    env = os.environ.copy()
    env.update({
        "OLLAMA_FLASH_ATTENTION": "1",
        "OLLAMA_KV_CACHE_TYPE": "q8_0",
        "OLLAMA_NUM_PARALLEL": "1",
        "OLLAMA_MAX_LOADED_MODELS": "1",
        "OLLAMA_KEEP_ALIVE": "30m",
        "OLLAMA_CONTEXT_LENGTH": str(context),
    })
    proc = subprocess.Popen(
        ["ollama", "serve"],
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    wait_http(f"{OLLAMA_URL}/api/tags")
    return proc

def stop_proc(proc):
    if proc is None or proc.poll() is not None:
        return
    proc.terminate()
    try:
        proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        proc.kill()
        proc.wait(timeout=5)

ollama_proc = start_ollama(HIGH_CONTEXT)
ACTIVE_CONTEXT = HIGH_CONTEXT

print("\nPulling model...", flush=True)
sh(["ollama", "pull", MODEL])

load = requests.post(
    f"{OLLAMA_URL}/api/generate",
    json={
        "model": MODEL,
        "prompt": "",
        "stream": False,
        "keep_alive": "30m",
        "options": {"num_ctx": ACTIVE_CONTEXT},
    },
    timeout=900,
)
load.raise_for_status()

ps = sh(["ollama", "ps"], capture=True)
ps_text = ps.stdout.strip()
print("\nollama ps:")
print(ps_text, flush=True)

if re.search(r"\bCPU\b", ps_text, flags=re.IGNORECASE):
    print(
        f"\nCPU placement detected. Retrying with {FALLBACK_CONTEXT:,} context...",
        flush=True,
    )
    stop_proc(ollama_proc)
    ACTIVE_CONTEXT = FALLBACK_CONTEXT
    ollama_proc = start_ollama(ACTIVE_CONTEXT)

    sh(["ollama", "pull", MODEL])
    load = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json={
            "model": MODEL,
            "prompt": "",
            "stream": False,
            "keep_alive": "30m",
            "options": {"num_ctx": ACTIVE_CONTEXT},
        },
        timeout=900,
    )
    load.raise_for_status()

    ps = sh(["ollama", "ps"], capture=True)
    print("\nollama ps after fallback:")
    print(ps.stdout.strip(), flush=True)

from flask import Flask, Response, jsonify, request, stream_with_context

app = Flask("toolboxlap")

@app.get("/health")
def health():
    return jsonify({
        "status": "ok",
        "service": "TOOLBOXLAP",
        "model": PUBLIC_MODEL_ID,
        "backend": MODEL,
        "context": ACTIVE_CONTEXT,
    })

@app.get("/v1/models")
def models():
    return jsonify({
        "object": "list",
        "data": [{
            "id": PUBLIC_MODEL_ID,
            "object": "model",
            "owned_by": "toolboxlap",
        }],
    })

@app.post("/v1/chat/completions")
def chat_completions():
    body = request.get_json(silent=True)
    if not isinstance(body, dict):
        return jsonify({"error": {"message": "JSON body required."}}), 400

    body["model"] = MODEL
    body["reasoning_effort"] = "none"

    if "max_tokens" not in body and "max_completion_tokens" not in body:
        body["max_tokens"] = 32768

    for key in [
        "tool_choice",
        "parallel_tool_calls",
        "store",
        "metadata",
        "service_tier",
        "logprobs",
        "top_logprobs",
        "modalities",
        "audio",
    ]:
        body.pop(key, None)

    if "max_completion_tokens" in body and "max_tokens" not in body:
        body["max_tokens"] = body["max_completion_tokens"]
    body.pop("max_completion_tokens", None)

    if isinstance(body.get("messages"), list):
        normalized = []
        for msg in body["messages"]:
            if isinstance(msg, dict):
                msg = dict(msg)
                if msg.get("role") == "developer":
                    msg["role"] = "system"
                if (
                    msg.get("role") == "assistant"
                    and msg.get("tool_calls")
                    and msg.get("content") == ""
                ):
                    msg["content"] = None
            normalized.append(msg)
        body["messages"] = normalized

    headers = {
        k: v
        for k, v in request.headers.items()
        if k.lower() not in {"host", "content-length", "connection"}
    }

    try:
        upstream = requests.post(
            f"{OLLAMA_URL}/v1/chat/completions",
            headers=headers,
            json=body,
            stream=True,
            timeout=900,
        )
    except requests.RequestException as e:
        return jsonify({"error": {"message": f"Ollama connection failed: {e}"}}), 502

    if upstream.status_code >= 400:
        print(
            f"\n⚠️ Ollama HTTP {upstream.status_code}: "
            f"{upstream.text[:3000]}",
            flush=True,
        )

    if body.get("stream", False):
        content_type = upstream.headers.get("content-type", "text/event-stream")

        @stream_with_context
        def generate():
            try:
                for chunk in upstream.iter_content(chunk_size=8192):
                    if chunk:
                        yield chunk
            finally:
                upstream.close()

        return Response(
            generate(),
            status=upstream.status_code,
            content_type=content_type,
        )

    data = upstream.content
    status = upstream.status_code
    content_type = upstream.headers.get("content-type", "application/json")
    upstream.close()

    return Response(
        data,
        status=status,
        content_type=content_type,
    )

def run_server():
    app.run(
        host="0.0.0.0",
        port=PROXY_PORT,
        debug=False,
        use_reloader=False,
    )

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
wait_http(f"http://127.0.0.1:{PROXY_PORT}/health", timeout=45)

if shutil.which("ngrok") is None:
    arch = "arm64" if os.uname().machine in {"aarch64", "arm64"} else "amd64"
    archive = f"ngrok-v3-stable-linux-{arch}.tgz"
    url = f"https://bin.equinox.io/c/bNyj1mQVY4c/{archive}"
    sh([
        "bash",
        "-lc",
        f"curl -fsSL {url} | tar xz -C /usr/local/bin ngrok",
    ])

if shutil.which("ngrok") is None:
    raise RuntimeError("ngrok installation failed.")

print("\nConfiguring ngrok...", flush=True)
subprocess.run(
    ["ngrok", "config", "add-authtoken", NGROK_AUTHTOKEN],
    check=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

ngrok_proc = subprocess.Popen(
    [
        "ngrok",
        "http",
        str(PROXY_PORT),
        "--host-header=rewrite",
        "--log=stdout",
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)

public_url = None
deadline = time.monotonic() + 45

while time.monotonic() < deadline:
    try:
        tunnels = requests.get(
            "http://127.0.0.1:4040/api/tunnels",
            timeout=2,
        ).json().get("tunnels", [])

        for tunnel in tunnels:
            url = tunnel.get("public_url", "")
            if url.startswith("https://"):
                public_url = url.rstrip("/")
                break

        if public_url:
            break

    except Exception:
        pass

    time.sleep(1)

if not public_url:
    stop_proc(ngrok_proc)
    stop_proc(ollama_proc)
    raise TimeoutError("ngrok did not publish an HTTPS tunnel.")

BASE_URL = public_url + "/v1"

print("\nTesting public API...", flush=True)

test = requests.post(
    public_url + "/v1/chat/completions",
    headers={
        "Content-Type": "application/json",
        "ngrok-skip-browser-warning": "true",
    },
    json={
        "model": PUBLIC_MODEL_ID,
        "messages": [{
            "role": "user",
            "content": "Reply with exactly: TOOLBOXLAP COLAB API WORKING",
        }],
        "stream": False,
        "max_tokens": 32,
    },
    timeout=300,
)

print("\n" + "=" * 72)
print("✅ TOOLBOXLAP COLAB PUBLIC API READY")
print("=" * 72)
print("\nCOPY THIS BASE URL INTO CLINE:")
print(BASE_URL)
print("\nCline Model ID:", PUBLIC_MODEL_ID)
print("Custom Header: ngrok-skip-browser-warning = true")
print("Backend model:", MODEL)
print("Active context:", ACTIVE_CONTEXT)
print("\nPublic API test HTTP:", test.status_code)

if test.ok:
    try:
        result = test.json()
        print(
            "Public API response:",
            result["choices"][0]["message"]["content"],
        )
    except Exception:
        print("Public API response: <non-JSON response>")

if not test.ok:
    print("\n⚠️ Public API test failed.")
    print(test.text[:5000])
else:
    print("\n✅ EVERYTHING IS WORKING")

print(
    "\nKeep this Colab cell running while the API is in use. "
    "Stop the runtime when finished."
)

try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("\nStopping TOOLBOXLAP...")
    stop_proc(ngrok_proc)
    stop_proc(ollama_proc)

